In [8]:
import warnings
warnings.filterwarnings("ignore")   # hide library notices

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD

# 1) The graph: each pair is an arrow (parent, child)
model = DiscreteBayesianNetwork(
    [
    ("heatwave", "overload"),
    ("heatwave", "powercut"),
    ("overload", "powercut"),
    ("powercut", "cancel"), 
    ]
)

# 2) P(Heatwave): no parents, so one column
heatwave = TabularCPD(
    variable = "heatwave", 
    variable_card = 2,
    values = [[0.3], [0.7]], #yes, #no
    state_names = {"heatwave": ["yes", "no"]},
)
    

# 3) P(Overload | Heatwave): one column for each value of heatwave
overload = TabularCPD(
    variable = "overload",
    variable_card = 2,
    values = [[0.6, 0.1],[0.4, 0.9]] ,
    evidence = ["heatwave"], 
    evidence_card = [2],
    state_names = {"heatwave": ["yes","no"], "overload": ["yes", "no"]},
)

# 4) P(Powercut | Heatwave, Overload): one column per (heatwave, overload) pair
powercut = TabularCPD(
    variable = "powercut",
    variable_card = 2,
    values = [[0.7, 0.2, 0.5, 0.05], [0.3, 0.8, 0.5, 0.95]],
    evidence = ["heatwave", "overload"],
    evidence_card = [2,2],
    state_names = {"heatwave": ["yes","no"], "overload": ["yes","no"], "powercut": ["yes","no"]},
)

#5 P(Cancel | Powercut): one column for each value of powercut
cancel = TabularCPD(
    variable = "cancel",
    variable_card = 2,
    values = [[0.8, 0.1], [0.2, 0.9]],
    evidence = ["powercut"],
    evidence_card = [2],
    state_names = {"powercut": ["yes", "no"], "cancel": ["yes","no"]},
)

model.add_cpds(heatwave,overload,powercut,cancel)
print("Model is correct", model.check_model())

Model is correct True


In [9]:
from pgmpy.inference import VariableElimination
inference = VariableElimination(model)

def show(evidence):
    """Print P(X | evidence) for every node X that we did not observe."""
    for node in ["heatwave", "overload", "powercut", "cancel"]:
        if node in evidence:
            print(f"{node}: {evidence[node]}   (observed)")
            continue
        result = inference.query([node], evidence=evidence, show_progress=False)
        print(node)
        for value, prob in zip(result.state_names[node], result.values):
            print(f"    {value}: {prob:.4f}")

show({"cancel": "yes"})

heatwave
    yes: 0.5367
    no: 0.4633
overload
    yes: 0.5474
    no: 0.4526
powercut
    yes: 0.6885
    no: 0.3115
cancel: yes   (observed)


In [ ]:
from pgmpy.factors.discrete import State
from pgmpy.sampling import BayesianModelSampling

sampler = BayesianModelSampling(model)
# Table of 10,000 samples with evidence cancel = yes
samples = sampler.rejection_sample(evidence=[State("cancel", "yes")], size=10000)
# Count the fraction of rows where heatwave is 'yes'
estimate = (samples["heatwave"] == "yes").mean()

print(f"Estimated P(heatwave = yes | cancel = yes): {estimate:.4f}")

100%|██████████| 10000/10000 [00:00<00:00, 29968.65it/s]

Estimated P(heatwave = yes | cancel = yes): 0.5400


The exact probability calculated for P(H = yes | L = yes) is 0.5367, whereas the rejection sampling estimate with 10,0000 samples produced an output of 0.5400. Both values are close because a large sample size results in the convergence of the answers towards the true probability.